# Brazilian E-Commerce Order and Delivery Analysis

## Business question

How do product mix, payment patterns, and delivery performance vary
across recorded Olist orders, and what practical areas merit further
investigation?

This project uses descriptive analysis. Results describe the supplied
records and do not establish causation, profit, or performance across
all Brazilian e-commerce.

## Dataset and attribution

Source: [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

Provided by Olist and distributed under
[CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/).

Original files are retained separately from analysis outputs.
Transformations, exclusions, and limitations are documented below.

### Imports and Paths

In [2]:
from pathlib import Path
from datetime import date
import hashlib
import json

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "olist"
ARCHIVE_PATH = (
    PROJECT_ROOT / "data" / "raw" / "olist_brazilian_ecommerce.zip"
)
DOCS_DIR = PROJECT_ROOT / "docs"
HASH_PATH = DOCS_DIR / "source_hashes.csv"

assert RAW_DIR.is_dir() , "Raw data folder missing. Check your notebook location."
assert ARCHIVE_PATH.is_file() , "Original ZIP file missing."
assert HASH_PATH.is_file() , "Run the previous hash-export step first."

print("Source paths found.")

Source paths found.


### Record the downloaded snapshot

In [3]:
RETRIEVED_ON = "2026-10-06"
DATASET_VERSION = "1"

VERSION_UPDATED_ON = ""

assert RETRIEVED_ON, "Enter the actual download date."
date.fromisoformat(RETRIEVED_ON)

assert DATASET_VERSION, "Enter the version selected for your download."

if VERSION_UPDATED_ON:
    date.fromisoformat(VERSION_UPDATED_ON)

SOURCE_URL = (
    "https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce"
)
LICENSE_URL = "https://creativecommons.org/licenses/by-nc-sa/4.0/"

### Check the File Inventory

In [4]:
EXPECTED_FILES = {
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
}

csv_paths = sorted(RAW_DIR.glob("*.csv"))
actual_files = {path.name for path in csv_paths}

missing_files = EXPECTED_FILES - actual_files
unexpected_files = actual_files - EXPECTED_FILES

assert not missing_files, f"Missing CSV files: {sorted(missing_files)}"
assert not unexpected_files, (
    f"Unexpected CSV files: {sorted(unexpected_files)}"
)

print(f"Verified {len(csv_paths)} expected CSV files.")

Verified 9 expected CSV files.


### Verify hashes and inspect schemas

In [5]:
def sha256_file(path):
    """Calculate a SHA-256 hash without loading the whole file into memory."""
    digest = hashlib.sha256()

    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024) , b""):
            digest.update(chunk)

    return digest.hexdigest()

recorded_hashes = pd.read_csv(
    HASH_PATH ,
    dtype='string' ,
    encoding='utf-8-sig'
)

assert set(recorded_hashes.columns) == {
    "filename", "Algorithm", "Hash"
}
assert not recorded_hashes["filename"].duplicated().any()
assert recorded_hashes["Algorithm"].eq("SHA256").all()

source_paths ={
    path.name: path
    for path in {ARCHIVE_PATH , *csv_paths}
}

assert set(recorded_hashes['filename']) == set(source_paths)

hash_lookup = recorded_hashes.set_index("filename")["Hash"]

file_metadata = []
audit_rows = []

for filename , path in source_paths.items():
    actual_hash = sha256_file(path)
    expected_hash = hash_lookup.loc[filename].lower()

    assert actual_hash == expected_hash , (
        f"Hash mismatch: {filename}. Investigate before continuing."
    )

    columns = (
        pd.read_csv(path , nrows=0 , encoding='utf-8-sig').columns.tolist()
        if path.suffix.lower() == '.csv'
        else None
    )

    columns = (
        pd.read_csv(path, nrows=0, encoding="utf-8-sig").columns.tolist()
        if path.suffix.lower() == ".csv"
        else None
    )

    file_metadata.append({
        "filename": filename,
        "relative_path": path.relative_to(PROJECT_ROOT).as_posix(),
        "bytes": path.stat().st_size,
        "sha256": actual_hash,
        "columns": columns,
    })

    audit_rows.append({
        "filename": filename,
        "bytes": path.stat().st_size,
        "column_count": len(columns) if columns is not None else None,
        "hash_matches": True,
    })

source_audit = pd.DataFrame(audit_rows)
source_audit["column_count"] = source_audit["column_count"].astype("Int64")

display(source_audit)

,filename,bytes,column_count,hash_matches
0,olist_orders_dataset.csv,17654914,8,True
1,olist_order_reviews_dataset.csv,14451670,7,True
2,olist_products_dataset.csv,2379446,9,True
3,olist_customers_dataset.csv,9033957,5,True
4,product_category_name_translation.csv,2613,2,True
5,olist_brazilian_ecommerce.zip,44717580,<NA>,True
6,olist_geolocation_dataset.csv,61273883,5,True
7,olist_order_payments_dataset.csv,5777138,5,True
8,olist_order_items_dataset.csv,15438671,7,True
9,olist_sellers_dataset.csv,174703,4,True


### Save the provenance manifest

In [6]:
manifest = {
    "dataset_title": "Brazilian E-Commerce Public Dataset by Olist",
    "attribution": "Olist",
    "source_url": SOURCE_URL,
    "dataset_version": DATASET_VERSION,
    "version_updated_on": VERSION_UPDATED_ON or None,
    "retrieved_on": RETRIEVED_ON,
    "license": "CC BY-NC-SA 4.0",
    "license_url": LICENSE_URL,
    "hash_algorithm": "SHA-256",
    "files": file_metadata,
}

manifest_path = DOCS_DIR / "data_manifest.json"

manifest_path.write_text(
    json.dumps(manifest , indent=2 , ensure_ascii=False) ,
    encoding='utf-8'
)

print("Saved provenance metadata to docs/data_manifest.json")

Saved provenance metadata to docs/data_manifest.json


## Initial source-table audit

Each table is inspected at its own row grain.

Orders, item lines, payment records, and review records are counted
separately. Multiple item, payment, or review records per order may
be legitimate.

Only fields needed for analysis and validation are loaded. Identifiers
remain internal join keys and are excluded from displayed outputs.

All rows are retained. Repeated rows in the selected columns are not
automatically treated as duplicates of complete source records.

### Load the seven relevant tables

In [7]:
TABLE_FILES = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "translations": "product_category_name_translation.csv",
}

COLUMNS = {
    "orders": [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "items": [
        "order_id",
        "order_item_id",
        "product_id",
        "price",
        "freight_value",
    ],
    "payments": [
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value",
    ],
    "reviews": [
        "review_id",
        "order_id",
        "review_score",
        "review_creation_date",
        "review_answer_timestamp",
    ],
    "customers": [
        "customer_id",
        "customer_state",
    ],
    "products": [
        "product_id",
        "product_category_name",
    ],
    "translations": [
        "product_category_name",
        "product_category_name_english",
    ],
}

raw_tables = {}

for name, filename in TABLE_FILES.items():
    columns = COLUMNS[name]

    raw_tables[name] = pd.read_csv(
        RAW_DIR / filename,
        usecols=columns,
        dtype="string",
        encoding="utf-8-sig",
        keep_default_na=False,
        skip_blank_lines=False,
        on_bad_lines="error",
    ).loc[:, columns]

print(f"Loaded {len(raw_tables)} tables.")

Loaded 7 tables.


### Audit missing and blank fields

In [8]:
def blank_mask(series):
    # identify empty or whitespace-only strings without changing values
    return series.str.strip().eq("").fillna(False)

field_rows = []
unavailable_masks = {}

for name , df in raw_tables.items():
    # keep missing values and blank string different
    missing = df.isna()
    blank = df.apply(blank_mask)
    unavailable = missing | blank

    unavailable_masks[name] = unavailable

    for column in df.columns:
        surrounding_whitespace = (
            df[column].str.strip().ne(df[column]).fillna(False)
            & ~blank[column]
        )

        unavailable_count = int(unavailable[column].sum())

        field_rows.append({
            "table": name,
            "column": column,
            "missing_values": int(missing[column].sum()),
            "blank_strings": int(blank[column].sum()),
            "missing_or_blank": unavailable_count,
            "missing_or_blank_pct": (
                round(100 * unavailable_count / len(df), 2)
                if len(df)
                else pd.NA
            ),
            "nonblank_values_with_surrounding_whitespace": (
                int(surrounding_whitespace.sum())
            ),
        })

field_audit = pd.DataFrame(field_rows)

with pd.option_context("display.max_rows" , None):
    display(field_audit)

,table,column,missing_values,blank_strings,missing_or_blank,missing_or_blank_pct,nonblank_values_with_surrounding_whitespace
0,orders,order_id,0,0,0,0.00,0
1,orders,customer_id,0,0,0,0.00,0
2,orders,order_status,0,0,0,0.00,0
3,orders,order_purchase_timestamp,0,0,0,0.00,0
4,orders,order_approved_at,0,160,160,0.16,0
5,orders,order_delivered_carrier_date,0,1783,1783,1.79,0
6,orders,order_delivered_customer_date,0,2965,2965,2.98,0
7,orders,order_estimated_delivery_date,0,0,0,0.00,0
8,items,order_id,0,0,0,0.00,0
9,items,order_item_id,0,0,0,0.00,0


### Audit candidate keys and repeated rows

In [13]:
KEY_COLUMNS = {
    "orders": ["order_id"],
    "items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"],
    "reviews": ["order_id", "review_id"],
    "customers": ["customer_id"],
    "products": ["product_id"],
    "translations": ["product_category_name"],
}

key_rows = []

for name , df in raw_tables.items():
    key_columns = KEY_COLUMNS[name]
    unavailable = unavailable_masks[name]

    incomplete_key = unavailable[key_columns].any(axis=1)

    # temporary key-only view for diagnostics; source rows remain unchanged
    complete_keys = df.loc[~incomplete_key , key_columns]

    participating = complete_keys.duplicated(keep=False)
    excess = complete_keys.duplicated(keep='first')

    duplicate_groups = len(
        complete_keys.loc[participating].drop_duplicates()
    )

    key_rows.append({
        "table": name,
        "rows": len(df),
        "loaded_columns": len(df.columns),
        "fully_empty_selected_rows": (
            int(unavailable.all(axis=1).sum())
        ),
        "rows_with_incomplete_key": int(incomplete_key.sum()),
        "duplicate_key_groups": duplicate_groups,
        "rows_in_duplicate_key_groups": int(participating.sum()),
        "excess_key_rows": int(excess.sum()),
        "repeated_selected_rows": int(df.duplicated().sum()),
    })

    key_audit = pd.DataFrame(key_rows)

    display(key_audit)

,table,rows,loaded_columns,fully_empty_selected_rows,rows_with_incomplete_key,duplicate_key_groups,rows_in_duplicate_key_groups,excess_key_rows,repeated_selected_rows
0,orders,99441,8,0,0,0,0,0,0


,table,rows,loaded_columns,fully_empty_selected_rows,rows_with_incomplete_key,duplicate_key_groups,rows_in_duplicate_key_groups,excess_key_rows,repeated_selected_rows
0,orders,99441,8,0,0,0,0,0,0
1,items,112650,5,0,0,0,0,0,0


,table,rows,loaded_columns,fully_empty_selected_rows,rows_with_incomplete_key,duplicate_key_groups,rows_in_duplicate_key_groups,excess_key_rows,repeated_selected_rows
0,orders,99441,8,0,0,0,0,0,0
1,items,112650,5,0,0,0,0,0,0
2,payments,103886,5,0,0,0,0,0,0


,table,rows,loaded_columns,fully_empty_selected_rows,rows_with_incomplete_key,duplicate_key_groups,rows_in_duplicate_key_groups,excess_key_rows,repeated_selected_rows
0,orders,99441,8,0,0,0,0,0,0
1,items,112650,5,0,0,0,0,0,0
2,payments,103886,5,0,0,0,0,0,0
3,reviews,99224,5,0,0,0,0,0,0


,table,rows,loaded_columns,fully_empty_selected_rows,rows_with_incomplete_key,duplicate_key_groups,rows_in_duplicate_key_groups,excess_key_rows,repeated_selected_rows
0,orders,99441,8,0,0,0,0,0,0
1,items,112650,5,0,0,0,0,0,0
2,payments,103886,5,0,0,0,0,0,0
3,reviews,99224,5,0,0,0,0,0,0
4,customers,99441,2,0,0,0,0,0,0


,table,rows,loaded_columns,fully_empty_selected_rows,rows_with_incomplete_key,duplicate_key_groups,rows_in_duplicate_key_groups,excess_key_rows,repeated_selected_rows
0,orders,99441,8,0,0,0,0,0,0
1,items,112650,5,0,0,0,0,0,0
2,payments,103886,5,0,0,0,0,0,0
3,reviews,99224,5,0,0,0,0,0,0
4,customers,99441,2,0,0,0,0,0,0
5,products,32951,2,0,0,0,0,0,0


,table,rows,loaded_columns,fully_empty_selected_rows,rows_with_incomplete_key,duplicate_key_groups,rows_in_duplicate_key_groups,excess_key_rows,repeated_selected_rows
0,orders,99441,8,0,0,0,0,0,0
1,items,112650,5,0,0,0,0,0,0
2,payments,103886,5,0,0,0,0,0,0
3,reviews,99224,5,0,0,0,0,0,0
4,customers,99441,2,0,0,0,0,0,0
5,products,32951,2,0,0,0,0,0,0
6,translations,71,2,0,0,0,0,0,0


- *Duplicate key groups*: how many complete key combinations repeat.
- *Rows in duplicate key groups*: all rows participating in those groups.
- *Excess key rows*: occurrences beyond the first.
- *Repeated selected rows*: occurrences identical across the columns we loaded.

### Inspect record per order

This shows why item lines, payment records, and reviews need separate treatment before joining. Multiple records per order are different from duplicate composite keys.

In [14]:
multiplicity_rows = []

for name in ['items' , 'payments' , 'reviews']:
    df = raw_tables[name]

    usable_order_key = ~unavailable_masks[name]['order_id']
    records_per_order = (
        df.loc[usable_order_key , 'order_id'].value_counts()
    )

    multiplicity_rows.append({
        "table": name,
        "distinct_orders_with_records": len(records_per_order),
        "orders_with_multiple_records": (
            int(records_per_order.gt(1).sum())
        ),
        "maximum_records_per_order": (
            int(records_per_order.max())
            if not records_per_order.empty
            else 0
        ),
        "rows_without_usable_order_key": (
            int((~usable_order_key).sum())
        ),
    })

multiplicity_audit = pd.DataFrame(multiplicity_rows)
display(multiplicity_audit)

,table,distinct_orders_with_records,orders_with_multiple_records,maximum_records_per_order,rows_without_usable_order_key
0,items,98666,9803,21,0
1,payments,99440,2961,29,0
2,reviews,98673,547,3,0


## Type conversion and reference checks

Empty and whitespace-only fields become missing values in working
copies. Source tables remain unchanged.

Nonblank values that fail conversion are counted separately from
source missingness. Nonfinite numeric values are also flagged.

Timestamps remain as recorded, without assigning a timezone.
No rows are removed.

Reference checks measure whether child keys appear in parent tables.
Successful matching alone does not establish safe join cardinality.

### Prepare working copies and conversion rules

In [16]:
import numpy as np

tables = {
    name: df.mask(unavailable_masks[name]).copy()
    for name , df in raw_tables.items()
}

DATE_COLUMNS = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp",
    ],
}

NUMERIC_COLUMNS = {
    "items": [
        "order_item_id",
        "price",
        "freight_value",
    ],
    "payments": [
        "payment_sequential",
        "payment_installments",
        "payment_value",
    ],
    "reviews": [
        "review_score",
    ],
}

conversion_rows = []
conversion_flags = {}

### Parse timestamps

In [17]:
for name , columns in DATE_COLUMNS.items():
    for column in columns:
        source = tables[name][column]

        parsed = pd.to_datetime(
            source ,
            format="%Y-%m-%d %H:%M:%S" ,
            exact=True ,
            errors='coerce'
        )

        invalid = source.notna() & parsed.isna()

        # retain row-leve diagnostics flag internally
        conversion_flags[(name , column)] = invalid

        conversion_rows.append({
            "table": name,
            "column": column,
            "type": "timestamp",
            "rows": len(source),
            "source_missing_or_blank": int(source.isna().sum()),
            "unparseable_nonblank": int(invalid.sum()),
            "nonfinite_numeric": 0,
            "valid_parsed_values": int(parsed.notna().sum()),
        })

        tables[name][column] = parsed

*errors="coerce"* represents parsing failures as NaT; the separate flags and counts prevent those failures from disappearing into general missingness.

### Parse numeric fields

In [18]:
for name , columns in NUMERIC_COLUMNS.items():
    for column in columns:
        source = tables[name][column]

        parsed = pd.to_numeric(source , errors='coerce')
        finite = np.isfinite(parsed).fillna(False)

        unparseable = source.notna() & parsed.isna()
        nonfinite = source.notna() & parsed.notna() & ~finite
        invalid = unparseable | nonfinite

        conversion_flags[(name , column)] = invalid

        conversion_rows.append({
            "table": name,
            "column": column,
            "type": "numeric",
            "rows": len(source),
            "source_missing_or_blank": int(source.isna().sum()),
            "unparseable_nonblank": int(unparseable.sum()),
            "nonfinite_numeric": int(nonfinite.sum()),
            "valid_parsed_values": int(finite.sum()),
        })

        # nonfinite values become unavailable in the working column
        tables[name][column] = parsed.mask(nonfinite)

conversion_audit= pd.DataFrame(conversion_rows)

partition_total = conversion_audit[
    [
        "source_missing_or_blank",
        "unparseable_nonblank",
        "nonfinite_numeric",
        "valid_parsed_values",
    ]
].sum(axis=1)

assert partition_total.eq(conversion_audit['rows']).all()

for name in tables:
    assert len(tables[name]) == len(raw_tables[name])

with pd.option_context("display.max_rows" , None):
    display(conversion_audit)

,table,column,type,rows,source_missing_or_blank,unparseable_nonblank,nonfinite_numeric,valid_parsed_values
0,orders,order_purchase_timestamp,timestamp,99441,0,0,0,99441
1,orders,order_approved_at,timestamp,99441,160,0,0,99281
2,orders,order_delivered_carrier_date,timestamp,99441,1783,0,0,97658
3,orders,order_delivered_customer_date,timestamp,99441,2965,0,0,96476
4,orders,order_estimated_delivery_date,timestamp,99441,0,0,0,99441
5,reviews,review_creation_date,timestamp,99224,0,0,0,99224
6,reviews,review_answer_timestamp,timestamp,99224,0,0,0,99224
7,items,order_item_id,numeric,112650,0,0,0,112650
8,items,price,numeric,112650,0,0,0,112650
9,items,freight_value,numeric,112650,0,0,0,112650


### Inspect numeric ranges and domain issues

In [19]:
INTEGER_FIELDS = {
    "order_item_id",
    "payment_sequential",
    "payment_installments",
    "review_score",
}

SEQUENCE_FIELDS = {
    "order_item_id",
    "payment_sequential",
}

numeric_rows = []

for name , columns in NUMERIC_COLUMNS.items():
    for column in columns:
        values = tables[name][column]
        valid_values = values.dropna()

        numeric_rows.append({
            "table": name,
            "column": column,
            "available_values": len(valid_values),
            "minimum": (
                valid_values.min() if len(valid_values) else pd.NA
            ),
            "maximum": (
                valid_values.max() if len(valid_values) else pd.NA
            ),
            "negative_values": int(values.lt(0).fillna(False).sum()),
            "zero_values": int(values.eq(0).fillna(False).sum()),
            "fractional_integer_field_values": (
                int(values.mod(1).ne(0).fillna(False).sum())
                if column in INTEGER_FIELDS
                else pd.NA
            ),
            "nonpositive_sequence_values": (
                int(values.le(0).fillna(False).sum())
                if column in SEQUENCE_FIELDS
                else pd.NA
            ),
            "scores_outside_1_to_5": (
                int(
                    (
                        values.notna()
                        & ~values.between(1, 5).fillna(False)
                    ).sum()
                )
                if column == "review_score"
                else pd.NA
            ),
        })

numeric_audit = pd.DataFrame(numeric_rows)
display(numeric_audit)

,table,column,available_values,minimum,maximum,negative_values,zero_values,fractional_integer_field_values,nonpositive_sequence_values,scores_outside_1_to_5
0,items,order_item_id,112650,1.00,21.00,0,0,0,0,<NA>
1,items,price,112650,0.85,6735.00,0,0,<NA>,<NA>,<NA>
2,items,freight_value,112650,0.00,409.68,0,383,<NA>,<NA>,<NA>
3,payments,payment_sequential,103886,1.00,29.00,0,0,0,0,<NA>
4,payments,payment_installments,103886,0.00,24.00,0,2,0,<NA>,<NA>
5,payments,payment_value,103886,0.00,13664.08,0,9,<NA>,<NA>,<NA>
6,reviews,review_score,99224,1.00,5.00,0,0,0,<NA>,0


- Item and payment sequences should be positive integers.
- Review scores should be integers from 1 to 5.
- Zero installments deserve investigation.
- Negative amounts deserve investigation.
- Zero freight or other zero amounts are not automatically errors.

### Check references between tables

In [21]:
RELATIONSHIPS = [
    ("items", "order_id", "orders", "order_id"),
    ("payments", "order_id", "orders", "order_id"),
    ("reviews", "order_id", "orders", "order_id"),
    ("orders", "customer_id", "customers", "customer_id"),
    ("items", "product_id", "products", "product_id"),
    (
        "products",
        "product_category_name",
        "translations",
        "product_category_name",
    ),
]

reference_rows = []

for child_name , child_column , parent_name , parent_column in RELATIONSHIPS:
    child_keys = tables[child_name][child_column]
    parent_keys = tables[parent_name][parent_column]

    present = child_keys.notna()

    # missing parent keys cannot act as valid matches
    matched = present & child_keys.isin(parent_keys.dropna())
    unmatched = present & ~matched

    present_count = int(present.sum())

    assert (
        int(child_keys.isna().sum())
        + int(matched.sum())
        + int(unmatched.sum())
        == len(child_keys)
    )

    reference_rows.append({
        "relationship": f"{child_name} → {parent_name}",
        "child_key": child_column,
        "child_rows": len(child_keys),
        "missing_child_key_rows": int(child_keys.isna().sum()),
        "unmatched_child_rows": int(unmatched.sum()),
        "unmatched_distinct_keys": child_keys.loc[unmatched].nunique(),
        "unmatched_pct_of_present_keys": (
            round(100 * int(unmatched.sum()) / present_count, 2)
            if present_count
            else pd.NA
        ),
        "missing_parent_key_rows": int(parent_keys.isna().sum()),
        "parent_excess_key_rows": (
            int(parent_keys.dropna().duplicated().sum())
        ),
    })

reference_audit = pd.DataFrame(reference_rows)
display(reference_audit)

,relationship,child_key,child_rows,missing_child_key_rows,unmatched_child_rows,unmatched_distinct_keys,unmatched_pct_of_present_keys,missing_parent_key_rows,parent_excess_key_rows
0,items → orders,order_id,112650,0,0,0,0.00,0,0
1,payments → orders,order_id,103886,0,0,0,0.00,0,0
2,reviews → orders,order_id,99224,0,0,0,0.00,0,0
3,orders → customers,customer_id,99441,0,0,0,0.00,0,0
4,items → products,product_id,112650,0,0,0,0.00,0,0
5,products → translations,product_category_name,32951,610,13,2,0.04,0,0


This displays counts rather than unmatched identifiers. Missing categories and untranslated nonblank categories are counted separately.